In [29]:
# Load the Kedro IPython extension
%load_ext kedro.ipython

The kedro.ipython extension is already loaded. To reload it, use:
  %reload_ext kedro.ipython


In [30]:
# Read parquet file into a DataFrame with polars
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [31]:
# df = pl.read_parquet(
#     "/Users/Carlos_Davalos/Downloads/w_0.8_d_60_o_3.0_t_0.01/second_stage_predictions.parquet"
# ).to_pandas()

df = catalog.load("second_stage_industrial_predictions").to_pandas()

df["corrected_predicted_value"] = df.apply(
    lambda row: row["predicted_value"]
    if row["predicted_value"] > row["c_yearly_margin_per_liter"]
    else row["c_yearly_margin_per_liter"],
    axis=1,
)

df["margin_change"] = df["corrected_predicted_value"] - df["c_yearly_margin_per_liter"]


def forced_decile_binning(series: pd.Series, prefix="Q"):
    # Perform quantile-based binning with duplicate edges dropped
    binned, bin_edges = pd.qcut(series, q=10, retbins=True, duplicates="drop")

    # Determine actual number of bins (may be < 10 if duplicates dropped)
    num_bins = len(bin_edges) - 1
    labels = [f"{prefix}{i + 1}" for i in range(num_bins)]

    # Re-bin using the known edges with labels applied
    binned = pd.cut(series, bins=bin_edges, labels=labels, include_lowest=True)

    # Create mapping from label to interval
    mapping = {labels[i]: (bin_edges[i], bin_edges[i + 1]) for i in range(num_bins)}

    return binned, mapping


# Apply to your DataFrame
df["margin_quantile"], quantile_label_map = forced_decile_binning(
    df["corrected_predicted_value"]
)


def assign_quantiles(df, column, quantiles=10):
    # Define labels dynamically based on column name
    labels = [f"Q{i + 1}" for i in range(quantiles - 1)]

    # Create the new column name
    new_col_name = f"{column}_quantile"

    # Assign quantiles
    df[new_col_name] = pd.qcut(df[column], q=quantiles, duplicates="drop")

    return df


df = assign_quantiles(df, "total_volume")

columns_to_quantile = [
    "planta_share_1627",
    "planta_share_1205",
    "planta_share_1210",
    "planta_share_1204",
    "planta_share_1228",
    "planta_share_1208",
    "planta_share_1212",
    "planta_share_1215",
    "planta_share_SIN INFORMACION",
    "planta_share_1211",
    "planta_share_1207",
    "planta_share_1206",
    "planta_share_1201",
    "planta_share_1202",
    "planta_share_1216",
    "planta_share_1214",
    "planta_share_1213",
    "planta_share_1203",
    "planta_share_1566",
]


# def apply_forced_binning(df, columns):
#     binning_mappings = {}

#     for col in columns:
#         prefix = "Q"
#         binned, mapping = forced_decile_binning(df[col], prefix=prefix)
#         df[f"{col}_quantile"] = binned
#         binning_mappings[col] = mapping

#     return df, binning_mappings


# # df, quantile_mappings = apply_forced_binning(df, columns_to_quantile)

[07/04/25 19:13:24] INFO     Loading data from second_stage_industrial_predictions              ]8;id=478206;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=437410;file:///Users/Carlos_Davalos/Projects/B2B_COPEC/b2b-pricing-model/.venv/lib/python3.11/site-packages/kedro/io/data_catalog.py#403\403]8;;\
                             (PolarsParquetDataset)...                                                             

In [32]:
for col in columns_to_quantile:
    df = assign_quantiles(df, col)

In [33]:
def format_interval_as_percent(interval):
    # Convert pandas.Interval to percentage string, e.g., (0.07, 0.11] → "7%–11%"
    left = f"{interval.left * 100:.0f}%"
    right = f"{interval.right * 100:.0f}%"
    return f"{left}–{right}"


def format_millions(value):
    return f"{value / 1000000:.1f}M" if value >= 1000000 else f"{value:,.0f}"


def format_volume_interval(interval):
    left = format_millions(interval.left)
    right = format_millions(interval.right)
    return f"{left}–{right}"

In [34]:
# Stats summary
total = len(df)
top_percent = (df["performance_label"] == "Top Performer").sum() / total * 100
under_percent = (df["performance_label"] == "Under Performer").sum() / total * 100
summary_text = f"✅ Top Performers: {top_percent:.1f}%<br>❌ Under Performers: {under_percent:.1f}%"

# Create vertical subplots (scatter on top, histogram below)
fig = make_subplots(
    rows=2,
    cols=1,
    row_heights=[0.6, 0.4],
    vertical_spacing=0.15,
    specs=[[{"type": "scatter"}], [{"type": "xy"}]],
)

# Scatter plot: predicted value
scatter_pred = px.scatter(
    df,
    x="c_yearly_margin_per_liter",
    y="corrected_predicted_value",
    size="total_volume",
    color="performance_label",
    size_max=20,
)
scatter_pred.update_traces(marker=dict(line=dict(width=0)))
for trace in scatter_pred.data:
    fig.add_trace(trace, row=1, col=1)

# Scatter plot: corrected predicted value
scatter_corr = px.scatter(
    df,
    x="c_yearly_margin_per_liter",
    y="corrected_predicted_value",
    size="total_volume",
    color="performance_label",
    size_max=20,
)
scatter_corr.update_traces(
    marker=dict(line=dict(width=0), opacity=0.6, symbol="circle-open")
)

# Histogram: margin change
fig.add_trace(
    go.Histogram(
        x=df["margin_change"],
        nbinsx=30,
        marker_color="teal",
        opacity=0.75,
        showlegend=False,
        histnorm="probability",
    ),
    row=2,
    col=1,
)

# Annotations
fig.add_annotation(
    text="Predicted vs Actual Margin",
    x=0.5,
    y=1.08,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(size=16),
    xanchor="center",
)
fig.add_annotation(
    text="Relative Error Distribution",
    x=0.5,
    y=0.35,
    xref="paper",
    yref="paper",
    showarrow=False,
    font=dict(size=14),
    xanchor="center",
)

# Performance summary box
fig.add_annotation(
    text=summary_text,
    xref="paper",
    yref="paper",
    x=0.01,
    y=0.96,
    showarrow=False,
    align="left",
    bgcolor="rgba(255,255,255,0.9)",
    bordercolor="black",
    borderwidth=1,
    font=dict(size=12),
)

# Layout updates
fig.update_layout(
    height=700,
    title_text="Performance Overview",
    margin=dict(t=100, b=60, l=60, r=40),
    showlegend=True,
)

# Axis labels
fig.update_xaxes(title_text="Actual Margin (c_yearly_margin_per_liter)", row=1, col=1)
fig.update_yaxes(title_text="Predicted Margin", row=1, col=1)
fig.update_xaxes(title_text="Relative Margin Change", row=2, col=1)
fig.update_yaxes(title_text="Probability", row=2, col=1)

fig.show()

In [35]:
import dash
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Input, Output, dcc, html
from plotly.subplots import make_subplots

# Dummy data example (replace this with your real `df`)
# df = pd.read_csv("your_data.csv")
# Example structure (you'll replace with actual df)
# df["margin_quantile"] = pd.qcut(df["margin_change"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])

# df["margin_quantile"] = pd.qcut(df["margin_change"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])

# Assuming df is already available
# numeric_cols = [col for col in df.select_dtypes(include="number").columns]
numeric_cols = df.columns

app = dash.Dash(__name__)
app.title = "Interactive Dashboard"

# --- 1. Predicted vs. Actual Margin ---
total = len(df)
top_percent = (df["performance_label"] == "Top Performer").sum() / total * 100
under_percent = (df["performance_label"] == "Under Performer").sum() / total * 100
non_regular_percent = (
    (df["performance_label"] == "Non Regular Client").sum() / total * 100
)
impact = sum(df["margin_change"] * df["total_volume"]) * 0.0011
impact_millions = impact / 1_000_000

summary_text = (
    f"<b>📊 Total Impact: ${impact_millions:,.2f}M</b><br>"
    f"<b>📊 Total Records: {total}</b><br>"
    f"✅ Top Performers: {top_percent:.1f}%<br>"
    f"❌ Under Performers: {under_percent:.1f}%"
    f"<br>⚠️ Non Regular Clients: {non_regular_percent:.1f}%<br>"
)

scatter_pred = px.scatter(
    df,
    x="c_yearly_margin_per_liter",
    y="corrected_predicted_value",
    size="total_volume",
    color="performance_label",
    size_max=20,
    hover_data={
        "customer_id": True,
        "c_yearly_margin_per_liter": ":$,.0f",  # Format as currency
        "corrected_predicted_value": ":$,.0f",  # Format as currency, no decimals
        "total_volume": ":,.0f",  # Comma-separated large numbers
    },
)

scatter_pred.update_layout(
    title="Se Predice el Margen Anual por Litro. En caso de que la prediccion sea menor al margen anual por litro, se corrige a este ultimo.",
    xaxis_title="Margen Anual por Litro (Actual)",
    yaxis_title="Margen Anual por Litro (Prediccion)",
    bargap=0.05,
)

scatter_pred.update_traces(marker=dict(line=dict(width=0)))

scatter_pred.add_annotation(
    text=summary_text,
    xref="paper",
    yref="paper",
    x=0.01,
    y=0.99,
    showarrow=False,
    align="left",
    bgcolor="rgba(255,255,255,0.9)",
    bordercolor="black",
    borderwidth=1,
    font=dict(size=12),
)

# --- 2. Histogram of Margin Change ---
# Compute min and max with rounding for bin alignment
start = df["margin_change"].min() // 1 * 1
end = df["margin_change"].max() // 1 * 1 + 1

df["margin_binned"] = df["margin_change"].copy()

# Clip all values greater than 20 to exactly 20
df["margin_binned"] = df["margin_binned"].apply(lambda x: 20 if x > 20 else x)

fig_hist = go.Figure()

fig_hist.add_trace(
    go.Histogram(
        x=df["margin_binned"],
        xbins=dict(
            start=df["margin_binned"].min() // 1 * 1,
            end=25,  # Include overflow bin up to 25
            size=1,
        ),
        marker_color="teal",
        opacity=0.75,
        showlegend=False,
    )
)

fig_hist.update_layout(
    title="Cantidad de Clientes por Cambio de Margen (Prediccion - Actual), (Agrupamos >20)",
    xaxis_title="Cambio de Margen (Prediccion - Actual)",
    yaxis_title="Frecuencia",
    bargap=0.05,
)

# --- 3. Volume vs Margin Scatterplot ---
fig_volume_margin = px.scatter(
    df,
    x="total_volume",
    y="margin_change",
    color="performance_label",
    size_max=20,
    title="Volume vs. Margin Change",
    hover_data={
        "customer_id": True,
        "margin_change": ":$,.0f",  # Format as currency
        "total_volume": ":,.0f",  # Comma-separated large numbers
    },
)

fig_volume_margin.update_layout(
    title="Evaluacion de Clientes por Volumen Anual vs Cambio de Margen",
    xaxis_title="Volumen Anual (Litros)",
    yaxis_title="Cambio de Margen (Prediccion - Actual)",
    bargap=0.05,
)


# App layout
app.layout = html.Div(
    [
        html.H1("Reporte - Modelo B2B Pricing"),
        html.H2("1. Margen - Prediccion vs Actual"),
        dcc.Graph(figure=scatter_pred),
        html.H2("2. Histograma de Cambio de Margen"),
        dcc.Graph(figure=fig_hist),
        html.H2("3. Volumen (Litros) vs Cambio de Margen"),
        dcc.Graph(figure=fig_volume_margin),
        html.H2("4. Histograma por Cuartiles"),
        html.Div(
            [
                html.Label("Seleccionar columna de cuartiles:"),
                dcc.Dropdown(
                    id="single-quantile-col",
                    options=[
                        {"label": col, "value": col} for col in columns_to_quantile
                    ],
                    value=columns_to_quantile[0],
                ),
                html.Label("Seleccionar variable continua:"),
                dcc.Dropdown(
                    id="single-histogram-var",
                    options=[{"label": col, "value": col} for col in numeric_cols],
                    value="total_volume",
                ),
            ],
            style={"margin-bottom": "20px"},
        ),
        dcc.Graph(id="heatmap_avg_prediction"),
        dcc.Graph(id="heatmap_client_count"),
    ]
)


@app.callback(
    Output("heatmap_avg_prediction", "figure"),
    Output("heatmap_client_count", "figure"),
    Input("single-quantile-col", "value"),
    Input("single-histogram-var", "value"),  # Not used, but can trigger refresh
)
def update_heatmaps(selected_quantile_col, _):
    df_ = df[df["performance_label"] == "Top Performer"]
    df_copy = (
        df_[
            [
                "total_volume",
                "c_yearly_margin_per_liter",
                "corrected_predicted_value",
                selected_quantile_col,
            ]
        ]
        .dropna()
        .copy()
    )

    # Compute dynamic quantile bins
    try:
        # df_copy["volume_quantile"] = pd.qcut(df_copy["total_volume"], 10)
        # Step 1: Sort and compute cumulative volume
        df_sorted = df_copy.sort_values("total_volume").copy()
        df_sorted["cumsum_volume"] = df_sorted["total_volume"].cumsum()
        total_volume = df_sorted["total_volume"].sum()
        df_sorted["volume_share"] = df_sorted["cumsum_volume"] / total_volume

        # Step 2: Assign bin number based on cumulative volume
        df_sorted["volume_bin"] = (df_sorted["volume_share"] * 10).apply(
            np.floor
        ).astype(int) + 1
        df_sorted["volume_bin"] = df_sorted["volume_bin"].clip(upper=10)

        # Step 3: Create intervals for labels
        bin_edges_df = df_sorted.groupby("volume_bin")["total_volume"].agg(
            ["min", "max"]
        )
        bin_intervals = bin_edges_df.apply(
            lambda row: pd.Interval(left=row["min"], right=row["max"], closed="right"),
            axis=1,
        )

        # Step 4: Map bin number to interval and assign as Categorical
        df_sorted["volume_quantile"] = df_sorted["volume_bin"].map(bin_intervals)

        # Convert to ordered categorical so `.cat.*` works
        cat_type = pd.api.types.CategoricalDtype(
            categories=bin_intervals.tolist(), ordered=True
        )
        df_sorted["volume_quantile"] = df_sorted["volume_quantile"].astype(cat_type)

        # Step 5: Merge into df_copy
        df_copy = df_copy.merge(
            df_sorted[["total_volume", "volume_quantile"]],
            on="total_volume",
            how="left",
        )
        df_copy["decile"] = pd.qcut(
            df_copy[selected_quantile_col], 10, duplicates="drop"
        )
    except ValueError:
        return go.Figure(), go.Figure()  # Handle edge cases gracefully

    # Convert bin categories to strings and preserve order
    volume_categories = df_copy["volume_quantile"].cat.categories
    decile_categories = df_copy["decile"].cat.categories

    ordered_volume_labels = [format_volume_interval(c) for c in volume_categories]
    ordered_decile_labels = [format_interval_as_percent(c) for c in decile_categories]

    df_copy["volume_quantile"] = pd.Categorical(
        [format_volume_interval(c) for c in df_copy["volume_quantile"]],
        categories=ordered_volume_labels,
        ordered=True,
    )
    df_copy["quantile_group"] = pd.Categorical(
        [format_interval_as_percent(c) for c in df_copy["decile"]],
        categories=ordered_decile_labels,
        ordered=True,
    )

    # --- 1. Avg predicted value heatmap ---
    avg_df = (
        df_copy.groupby(["quantile_group", "volume_quantile"], observed=True)
        .agg(avg_value=("c_yearly_margin_per_liter", "mean"))
        .reset_index()
    )
    pivot_avg = avg_df.pivot(
        index="quantile_group", columns="volume_quantile", values="avg_value"
    )

    fig_avg = px.imshow(
        pivot_avg,
        labels=dict(
            x="Cuantil de Volumen", y="Decil de Grupo", color="Predicción Promedio"
        ),
        x=ordered_volume_labels,
        y=ordered_decile_labels,
        text_auto=".2f",
        aspect="auto",
    )
    fig_avg.update_layout(
        title=f"Predicción Promedio por Decil de {selected_quantile_col} y Cuantil de Volumen"
    )

    # --- 2. Client count heatmap ---
    count_df = (
        df_copy.groupby(["quantile_group", "volume_quantile"], observed=True)
        .size()
        .reset_index(name="client_count")
    )
    pivot_count = count_df.pivot(
        index="quantile_group", columns="volume_quantile", values="client_count"
    )

    fig_count = px.imshow(
        pivot_count,
        labels=dict(
            x="Cuantil de Volumen", y="Decil de Grupo", color="Cantidad de Clientes"
        ),
        x=ordered_volume_labels,
        y=ordered_decile_labels,
        text_auto=True,
        aspect="auto",
    )
    fig_count.update_layout(
        title=f"Cantidad de Clientes por Decil de {selected_quantile_col} y Cuantil de Volumen"
    )

    return fig_avg, fig_count


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=9191)

<IPython.lib.display.IFrame object at 0x31ad89690>